# 1 · The design space

A matrix–vector product on a compute-near-memory system: one rank of **64 DPUs**,
each with up to **16 tasklets** (hardware threads), each DPU owning a slice of
memory (MRAM) and a small working memory (WRAM).

To run `y = A·x` there, the loop nest has to be **tiled** so that every DPU gets a
piece of `A` that fits its MRAM, every tasklet a piece that fits WRAM, and the
pieces are laid out so the result can be gathered back. Which tile sizes, how
many tasklets, in which order the dimensions are distributed — those are the
*design parameters*. This notebook is about how many there are, and how few
combinations of them are even legal.

In [17]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dse
import dse.plots as plots
from dse import DATA, WORK

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

## The program

Cinnamon's input is MLIR. The kernel is a single `cinm.op.gemv` on a
1024 × 1024 matrix; the `#upmem.platform` attribute says what hardware is
available.

In [18]:
mlir = DATA / "small" / "gemv.mlir"
dse.show_mlir(mlir)

AttributeError: module 'dse' has no attribute 'show_mlir'

## Your guess

Before the tools tell us anything: pick a configuration by hand.

The picker only offers choices that can still complete a **feasible**
configuration — as you fix parameters, the other dropdowns shrink to what remains
legal. Fix all of them, then press **Simulate**: `eval-solution` runs the whole
lowering for that one configuration and reports what the simulator estimates —
a fraction of a second.

- `tasklets` — threads per DPU
- `gemv.M.mram`, `gemv.K.mram` — the tile of `A` each *(DPU, tasklet)* worker
  holds in its MRAM, along the two dimensions of the loop nest
  (M = rows of A / entries of y, K = columns of A / entries of x)
- `gemv.M.wram`, `gemv.K.wram` — the inner tile a tasklet streams through WRAM
- `gemv.order` — which dimension is spread across DPUs first

In [ ]:
import json

space = dse.load_space(DATA / "small" / "space.json")
feasible = dse.load_pool(DATA / "small" / "pool.csv")[lambda p: dse.param_columns(p)]


def simulate(config):
    report = dse.eval_solution(mlir, config)
    # Notebook 2 reads this back to show where the guess landed.
    (WORK / "my_guess.json").write_text(
        json.dumps({"config": config, "cost_ms": report.total_ms}, indent=1)
    )
    return report


picker = dse.ConfigPicker(feasible, space, on_pick=simulate).show()

The breakdown says where the time goes: the kernel itself, launching it,
moving the operands in and the result back. Keep the total in mind.

Skipped the picker? The cell below falls back to a plausible first guess —
all 16 tasklets, middling tiles — so the rest of the tutorial has something to
compare against.

In [ ]:
if picker.result is None:
    report = simulate(picker.default_guess())
    print("no configuration simulated in the widget; used a default guess:", report.config)
else:
    report = picker.result
report

In the next notebook you will see where that number lands among *all*
feasible configurations.

## Where the space comes from

The picker knew which combinations were legal. That knowledge is derived, not
listed: from the loop nest of the operation and the memory hierarchy of the
platform, Cinnamon builds a **constraint system** and solves it. `dump-space-only`
does exactly that and stops.

In [ ]:
space = dse.dump_space(mlir)
print(dse.cinm.describe_space(space))

Four billion combinations of parameter values, of which about three thousand
are configurations that can run at all. The parameters:

In [ ]:
dse.cinm.params_table(space)

And the constraints that cut the Cartesian product down. Some are pure
arithmetic on the problem shape and were applied as filters; the rest went to
the solver. Read them as the hardware talking: *a tile has to divide the
dimension*, *what the tasklets hold has to fit MRAM*, *a DMA moves 8 bytes at a
time*, *there is exactly one tile per worker*.

In [ ]:
pd.set_option("display.max_colwidth", 120)
dse.cinm.constraints_table(space)

**What to take away.** The design space is not something we wrote down; it
falls out of the program's structure and the hardware's structure. The same
derivation, with a different platform description, gives the space of a
different CNM system.

→ Notebook 2: what the 3234 configurations look like.